# Customer Purchase Propensity V2
## A CRISP-DM case study for weekly household prioritization

**Decision:** Khi capacity có hạn, hộ nào nên được ưu tiên vì có xác suất phát sinh ít nhất một basket trong 7 ngày tới cao nhất?

Notebook này là evidence trail: từ business framing, data audit, target validation, point-in-time features, rolling backtest đến deployment.

### Executive result
Temporal holdout: **PR-AUC 0.8506**, **ROC-AUC 0.8303**, **Brier 0.1679**, **lift@10% 1.819x**. Kết luận: go cho controlled prioritization pilot; propensity không chứng minh incremental effect của coupon.

In [ ]:
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix, precision_recall_curve, roc_curve)

pd.set_option('display.max_columns', 100)
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
EDA = ROOT / 'artifacts' / 'propensity_v2' / 'data_understanding'
MODEL = ROOT / 'artifacts' / 'propensity_v2' / 'model'
metrics = json.loads((MODEL / 'metrics_v2.json').read_text(encoding='utf-8'))
audit = json.loads((EDA / 'data_audit.json').read_text(encoding='utf-8'))
TARGET = 'target_purchase_next_7d'
metrics['selected_model'], metrics['test_metrics']

# Phase 1 – Business Understanding

Prediction unit là `(household_key, snapshot_day)`. Output gồm calibrated probability, rank và percentile. Primary use case là weekly prioritization, không phải tự động phát coupon.

Success gates: PR-AUC ≥ 0.83, lift@10% ≥ 1.70, Brier ≤ 0.18, có temporal holdout và không leakage.

## Why a 7-day horizon?
Horizon phải vừa đủ actionable và không biến hầu hết household thành positive.

In [ ]:
horizons = pd.read_csv(EDA / 'target_horizon_summary.csv')
horizons.style.format({'mean_target_rate': '{:.2%}', 'min_target_rate': '{:.2%}', 'max_target_rate': '{:.2%}'})

In [ ]:
ax = horizons.plot.bar(x='horizon_days', y='mean_target_rate', legend=False, color=['#2F5597', '#70AD47', '#ED7D31'], title='Target rate by prediction horizon')
ax.set_ylabel('Positive rate'); ax.set_xlabel('Horizon (days)')
ax.set_ylim(0, 1); plt.tight_layout()

**Decision:** giữ 7 ngày. Target rate 53.05% cân bằng hơn 14 ngày (68.26%) và 30 ngày (80.96%), đồng thời khớp weekly operating cadence.

# Phase 2 – Data Understanding

## Data landscape

In [ ]:
inventory = pd.read_csv(EDA / 'dataset_inventory.csv')
inventory[['file', 'size_mb', 'rows', 'columns', 'duplicate_rows', 'missing_cells']]

Tám bảng gồm transaction, product, demographic, campaign, coupon/redemption và 36.8 triệu dòng product-store-week media causal.

## Transaction integrity

In [ ]:
quality_keys = ['rows', 'households', 'baskets', 'products', 'stores', 'day_min', 'day_max', 'exact_duplicate_rows', 'multi_household_baskets', 'multi_day_baskets', 'invalid_trans_time_rows']
pd.Series({key: audit['transaction_quality'][key] for key in quality_keys}, name='value').to_frame()

## A data issue that changed the model

In [ ]:
pd.Series(audit['transaction_quality']['quantity_distribution'], name='QUANTITY').to_frame()

`QUANTITY` đạt 89,638 và chủ yếu bị scale ở KIOSK-GAS/MISC SALES. V2 loại feature này và dùng basket count, line items cùng diversity để tránh so sánh những đơn vị không tương thích.

## Product and department mix

In [ ]:
department = pd.read_csv(EDA / 'department_summary.csv').head(12)
department[['DEPARTMENT', 'line_items', 'households', 'sales', 'sales_share']]

In [ ]:
plot_data = department.sort_values('sales_share')
ax = plot_data.plot.barh(x='DEPARTMENT', y='sales_share', legend=False, title='Sales share by leading departments', color='#4472C4')
ax.set_xlabel('Sales share'); plt.tight_layout()

## Temporal behavior

In [ ]:
weekly = pd.read_csv(EDA / 'weekly_summary.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
weekly.plot(x='WEEK_NO', y='households', ax=axes[0], legend=False, title='Active households by week')
weekly.plot(x='WEEK_NO', y='sales', ax=axes[1], legend=False, title='Observed sales by week', color='#70AD47')
plt.tight_layout()

## Household heterogeneity

In [ ]:
households = pd.read_csv(EDA / 'household_summary.csv')
households[['active_days', 'baskets', 'spend', 'mean_gap_days', 'avg_basket_value']].describe(percentiles=[.01, .25, .5, .75, .99]).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
households['active_days'].clip(upper=households['active_days'].quantile(.99)).hist(bins=30, ax=axes[0]); axes[0].set_title('Active days (winsorized p99)')
households['mean_gap_days'].clip(upper=households['mean_gap_days'].quantile(.99)).hist(bins=30, ax=axes[1]); axes[1].set_title('Mean purchase gap')
households['avg_basket_value'].clip(upper=households['avg_basket_value'].quantile(.99)).hist(bins=30, ax=axes[2]); axes[2].set_title('Average basket value')
plt.tight_layout()

## Join coverage and promotion data

In [ ]:
pd.Series(audit['relational_integrity_and_coverage'], name='value').to_frame()

Product join đạt 100%; campaign household coverage 63.36%; demographic coverage 32.04%; redemption đều khớp campaign window. Thiếu demographic được giữ là `Unknown`, không loại khỏi cohort.

# Phase 3 – Data Preparation

## Point-in-time contract

- Feature window: `(snapshot-window, snapshot]`.
- Label window: `(snapshot, snapshot+7]`.
- Household chỉ eligible sau lần mua đầu tiên.
- Snapshot cách 7 ngày; embargo một horizon giữa partitions.
- Không dùng future campaign/redemption outcome.

In [ ]:
pd.DataFrame({key: value for key, value in metrics['split_manifest'].items() if isinstance(value, dict)}).T

## Feature architecture

In [ ]:
modeling_table = pd.read_parquet(MODEL / 'modeling_table_v2.parquet')
{'rows': len(modeling_table), 'columns': len(modeling_table.columns), 'duplicate_keys': int(modeling_table.duplicated(['household_key', 'snapshot_day']).sum()), 'missing_cells': int(modeling_table.isna().sum().sum()), 'target_rate': modeling_table[TARGET].mean()}

170 features: cadence, basket, monetary, promotion, category, brand, daypart, assortment/store diversity, lifecycle và demographics, trên các cửa sổ 7/30/90/180 ngày.

# Phase 4 – Modeling

Model family được chọn bằng ba expanding-window rolling-origin folds, không chọn theo test holdout.

In [ ]:
selection = pd.read_csv(MODEL / 'model_selection_summary.csv')
selection.style.format({'mean_pr_auc': '{:.4f}', 'std_pr_auc': '{:.4f}', 'mean_roc_auc': '{:.4f}', 'mean_brier_score': '{:.4f}', 'mean_lift_at_10pct': '{:.3f}'})

In [ ]:
backtest = pd.read_csv(MODEL / 'rolling_backtest_metrics.csv')
pivot = backtest.pivot(index='fold', columns='model', values='pr_auc')
ax = pivot.plot(marker='o', figsize=(10, 4), title='Rolling-origin PR-AUC')
ax.set_ylabel('PR-AUC'); ax.grid(alpha=.3); plt.tight_layout()

Regularized HGB thắng với mean PR-AUC 0.85885. Full logistic chỉ thấp hơn khoảng 0.0022 và được giữ làm challenger để cân bằng explainability/operations.

# Phase 5 – Evaluation

## Holdout metrics and uncertainty

In [ ]:
pd.Series(metrics['test_metrics'], name='test').to_frame()

In [ ]:
intervals = pd.read_csv(MODEL / 'bootstrap_confidence_intervals.csv')
intervals

## Calibration

In [ ]:
calibration = pd.read_csv(MODEL / 'test_calibration.csv')
ax = calibration.plot(x='predicted_probability', y='observed_rate', marker='o', xlim=(0,1), ylim=(0,1), grid=True, legend=False, title='Calibration by score decile')
ax.plot([0,1], [0,1], '--', color='gray'); plt.tight_layout()

## ROC and precision-recall curves

In [ ]:
predictions = pd.read_csv(MODEL / 'test_predictions.csv')
y = predictions[TARGET].to_numpy(); p = predictions['purchase_probability'].to_numpy()
fpr, tpr, _ = roc_curve(y, p); precision, recall, _ = precision_recall_curve(y, p)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr); axes[0].plot([0,1],[0,1],'--',color='gray'); axes[0].set(title='ROC curve', xlabel='FPR', ylabel='TPR')
axes[1].plot(recall, precision); axes[1].axhline(y.mean(),ls='--',color='gray'); axes[1].set(title='Precision-recall curve', xlabel='Recall', ylabel='Precision')
plt.tight_layout()

## Threshold view (diagnostic, not the default business rule)

In [ ]:
threshold = metrics['test_metrics']['threshold']
matrix = confusion_matrix(y, p >= threshold)
ConfusionMatrixDisplay(matrix).plot(cmap='Blues'); plt.title(f'Threshold={threshold:.3f}')

## Capacity curve: the management decision view

In [ ]:
topk = pd.read_csv(MODEL / 'top_k_business_metrics.csv')
topk.style.format({'population_fraction': '{:.0%}', 'precision': '{:.2%}', 'recall': '{:.2%}', 'lift': '{:.3f}'})

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(topk['population_fraction'], topk['precision'], marker='o', label='Precision')
ax1.plot(topk['population_fraction'], topk['recall'], marker='o', label='Recall')
ax1.set(xlabel='Population contacted', ylabel='Rate', title='Capacity trade-off')
ax1.legend(); ax1.grid(alpha=.3); plt.tight_layout()

## Stability by test snapshot

In [ ]:
stability = pd.read_csv(MODEL / 'test_metrics_by_snapshot.csv')
ax = stability.plot(x='snapshot_day', y=['pr_auc','roc_auc','brier_score'], marker='o', figsize=(10,4), title='Holdout stability across time')
ax.grid(alpha=.3); plt.tight_layout()

## Segment and error analysis

In [ ]:
segments = pd.read_csv(MODEL / 'test_metrics_by_segment.csv')
segments

Model yếu hơn ở nhóm activity trung bình và long-recency. Nhóm có demographic có purchase rate cao hơn nhiều, nhiều khả năng do selection bias; không diễn giải demographic là nguyên nhân.

## Explainability by feature family

In [ ]:
grouped = pd.read_csv(MODEL / 'grouped_feature_importance.csv')
grouped

In [ ]:
plot_grouped = grouped.sort_values('importance_pr_auc_drop')
ax = plot_grouped.plot.barh(x='feature_family', y='importance_pr_auc_drop', legend=False, color='#5B9BD5', title='Grouped permutation importance')
ax.set_xlabel('PR-AUC drop after joint permutation'); plt.tight_layout()

Cadence và assortment/store diversity chi phối. Individual features tương quan mạnh giữa các windows nên grouped importance đáng tin hơn cho narrative.

## Drift evidence

In [ ]:
drift = pd.read_csv(MODEL / 'feature_drift_psi.csv')
drift.head(15)

Tenure/lifetime drift là structural do snapshot về sau tích lũy nhiều lịch sử hơn; không dùng chúng làm fixed PSI alert. Rolling features mới là monitoring signal.

# Phase 6 – Deployment

## Latest batch score

In [ ]:
scores = pd.read_csv(MODEL / 'latest_scores_v2.csv')
scores.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
scores['purchase_probability_next_7d'].hist(bins=30, ax=axes[0])
axes[0].set_title('Latest score distribution')
scores.plot.scatter(x='rank', y='purchase_probability_next_7d', ax=axes[1], s=8)
axes[1].set_title('Probability by rank'); plt.tight_layout()

## Production contract

Output: household, snapshot, calibrated probability, high-propensity flag, rank và percentile. Monitor input contract, rolling-feature drift, calibration, PR-AUC/lift khi label mature, cùng incremental business KPIs.

## Recommendation

**Go:** controlled weekly prioritization pilot.

**No-go:** auto-coupon optimization cho đến khi có randomized treatment/control.

Pilot cần đo incremental purchase, contribution margin, contact cost và unsubscribe/complaint guardrails qua tối thiểu 6–8 chu kỳ.

# Reproducibility

```powershell
python src/audit_dunnhumby_data.py
python src/run_propensity_v2.py train
python src/run_propensity_v2.py score
python src/validate_propensity_v2.py
python -m pytest -q
```